In [ ]:
# burgers_theta_to_heatmaps_autoarch_jupyter_safe.py
# ------------------------------------------------------------
# Load saved Burgers theta files, rebuild the correct NN shape,
# evaluate u_theta(x,t) on the full burgers.mat grid, and save:
#   1) heatmap PNG: exact / prediction / |error|
#   2) result NPZ: x, t, u_true, u_pred, rel_l2, mse
#   3) summary CSV
#
# Handles your mixed architectures:
#   Adam:      [2, 50, 50, 50, 1]  theta size 5301
#   ALM/SQP:   [2, 30, 30, 30, 1]  theta size 1951
#
# Terminal:
#   python burgers_theta_to_heatmaps_autoarch_jupyter_safe.py . --mat data/burgers.mat --outdir burgers_theta_heatmaps
#
# Jupyter:
#   %run burgers_theta_to_heatmaps_autoarch_jupyter_safe.py . --mat data/burgers.mat --outdir burgers_theta_heatmaps
# ------------------------------------------------------------

import os
import math
import glob
import argparse
from pathlib import Path

import numpy as np
import scipy.io
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp


# -----------------------------
# Precision
# -----------------------------
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32


# -----------------------------
# Known architectures
# -----------------------------
ARCH_BY_BASENAME = {
    # Adam baseline: hidden_dim=50, num_hidden=3
    "burgers_adam_theta.npy": [2, 50, 50, 50, 1],

    # Traditional ALM: hidden_dim=30, num_hidden=3
    "theta_traditional_alm_burgers_lbfgs_20000_best.npy": [2, 30, 30, 30, 1],

    # SQP jittered / deterministic: hidden_dim=30, num_hidden=3
    "theta_star_pointwise_jitter_1e-5.npy": [2, 30, 30, 30, 1],
    "theta_star_pointwise_jitter_1e-5.npz": [2, 30, 30, 30, 1],
    "theta_star_pointwise_jitter_1e-5.npz.npy": [2, 30, 30, 30, 1],

    "theta_star_pointwise_jitter_deterministic.npy": [2, 30, 30, 30, 1],
    "theta_star_pointwise_jitter_deterministic.npz": [2, 30, 30, 30, 1],
    "theta_star_pointwise_jitter_deterministic.npz.npy": [2, 30, 30, 30, 1],
}


# Fallback by theta vector size.
# 2-50-50-50-1 = 5301
# 2-30-30-30-1 = 1951
ARCH_BY_THETA_SIZE = {
    5301: [2, 50, 50, 50, 1],
    1951: [2, 30, 30, 30, 1],
}


# -----------------------------
# Burgers data loading
# -----------------------------
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)

    t = np.asarray(d["t"]).squeeze()
    x = np.asarray(d["x"]).squeeze()
    usol = np.asarray(d["usol"])

    # Force usol to shape (nt, nx)
    if usol.shape == (len(x), len(t)):
        usol = usol.T

    if usol.shape != (len(t), len(x)):
        raise ValueError(
            f"Bad usol shape {usol.shape}; expected {(len(t), len(x))} "
            f"or {(len(x), len(t))}."
        )

    nu = float(np.asarray(d["nu"]).squeeze()) if "nu" in d else np.nan
    return t, x, usol, nu


def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


# -----------------------------
# MLP utilities
# -----------------------------
def shapes_from_layer_sizes(layer_sizes):
    shapes = []
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        shapes.append(((m, n), (n,)))
    return tuple(shapes)


def expected_theta_size(layer_sizes):
    return sum(m * n + n for m, n in zip(layer_sizes[:-1], layer_sizes[1:]))


def unflatten_params(theta, shapes):
    theta = jnp.asarray(theta, dtype=DTYPE).reshape(-1)
    params = []
    idx = 0

    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(
            f"Theta size mismatch: used {idx}, theta has {theta.size}. "
            f"Check architecture."
        )

    return params


def mlp_apply(params, X):
    h = X
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h


def predict_u(theta, layer_sizes, X_grid):
    shapes = shapes_from_layer_sizes(layer_sizes)
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X_grid)[:, 0]


# -----------------------------
# Theta loading
# -----------------------------
def load_theta(path):
    obj = np.load(path, allow_pickle=True)

    # Normal .npy file
    if isinstance(obj, np.ndarray):
        return np.asarray(obj).reshape(-1)

    # .npz file
    keys = list(obj.keys())

    for key in ["theta", "theta_star", "params", "arr_0"]:
        if key in keys:
            return np.asarray(obj[key]).reshape(-1)

    raise KeyError(f"Could not find theta array in {path}. Keys = {keys}")


def infer_arch(path, theta):
    base = Path(path).name

    if base in ARCH_BY_BASENAME:
        arch = ARCH_BY_BASENAME[base]
        if expected_theta_size(arch) == theta.size:
            return arch

    if theta.size in ARCH_BY_THETA_SIZE:
        return ARCH_BY_THETA_SIZE[int(theta.size)]

    raise ValueError(
        f"Cannot infer architecture for {path}. "
        f"Theta size = {theta.size}. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME."
    )


def is_theta_candidate(path):
    base = Path(path).name.lower()

    # Exclude histories and already-created result files.
    bad_tokens = [
        "hist_",
        "history",
        "result",
        "heatmap",
        "summary",
        "advection",
        "moose",
        "reference",
        "dataset",
    ]
    if any(tok in base for tok in bad_tokens):
        return False

    # Include likely theta files.
    good_tokens = [
        "theta",
        "burgers_adam_theta",
        "theta_star",
    ]

    if any(tok in base for tok in good_tokens):
        return path.endswith((".npy", ".npz", ".npz.npy"))

    return False


def collect_theta_files(paths):
    files = []

    for p in paths:
        p = str(p)

        if os.path.isdir(p):
            for ext in ["*.npy", "*.npz", "*.npz.npy"]:
                files.extend(glob.glob(os.path.join(p, "**", ext), recursive=True))

        elif os.path.isfile(p):
            files.append(p)

        else:
            files.extend(glob.glob(p, recursive=True))

    files = sorted(set(files))
    files = [f for f in files if is_theta_candidate(f)]
    return files


# -----------------------------
# Evaluation and plotting
# -----------------------------
def eval_theta_on_burgers(theta, layer_sizes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.asarray(X_grid_np, dtype=DTYPE)

    u_pred = np.asarray(predict_u(theta, layer_sizes, X_grid)).reshape(grid_shape)
    u_true = np.asarray(usol_np, dtype=np.float64)

    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))

    return mse, rel_l2, u_pred


def clean_stem(path):
    name = Path(path).name
    for suffix in [".npz.npy", ".npy", ".npz"]:
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return Path(path).stem


def plot_heatmap_triplet(x_np, t_np, u_true, u_pred, title, save_path):
    abs_err = np.abs(u_pred - u_true)

    vmin = min(float(np.min(u_true)), float(np.min(u_pred)))
    vmax = max(float(np.max(u_true)), float(np.max(u_pred)))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

    im0 = axes[0].pcolormesh(x_np, t_np, u_true, shading="auto", vmin=vmin, vmax=vmax)
    axes[0].set_title("Exact")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("t")
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].pcolormesh(x_np, t_np, u_pred, shading="auto", vmin=vmin, vmax=vmax)
    axes[1].set_title("Prediction")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("t")
    fig.colorbar(im1, ax=axes[1])

    im2 = axes[2].pcolormesh(x_np, t_np, abs_err, shading="auto")
    axes[2].set_title("|Error|")
    axes[2].set_xlabel("x")
    axes[2].set_ylabel("t")
    fig.colorbar(im2, ax=axes[2])

    fig.suptitle(title, fontsize=13)

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def process_all(paths, mat_path="data/burgers.mat", outdir="burgers_theta_heatmaps"):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    print(f"Loading Burgers data: {mat_path}")
    t_np, x_np, usol_np, nu = load_burgers_mat(mat_path)
    print(f"Data: t={t_np.shape}, x={x_np.shape}, usol={usol_np.shape}, nu={nu}")

    theta_files = collect_theta_files(paths)
    print(f"Found {len(theta_files)} theta candidate files.")

    if len(theta_files) == 0:
        print("No theta files found.")
        print("Expected examples:")
        print("  burgers_adam_theta.npy")
        print("  theta_traditional_alm_burgers_lbfgs_20000_best.npy")
        print("  theta_star_pointwise_jitter_1e-5.npy")
        print("  theta_star_pointwise_jitter_deterministic.npy")
        return []

    summary = []

    for path in theta_files:
        try:
            theta = load_theta(path)
            arch = infer_arch(path, theta)

            expected = expected_theta_size(arch)
            if theta.size != expected:
                print(f"[SKIP] {path}")
                print(f"  theta size {theta.size} != expected {expected} for arch {arch}")
                continue

            mse, rel_l2, u_pred = eval_theta_on_burgers(theta, arch, x_np, t_np, usol_np)

            stem = clean_stem(path)
            heatmap_path = outdir / f"{stem}_heatmap.png"
            result_path = outdir / f"{stem}_result.npz"

            title = f"{stem} | arch={arch} | relL2={rel_l2:.3e}"

            plot_heatmap_triplet(
                x_np=x_np,
                t_np=t_np,
                u_true=usol_np,
                u_pred=u_pred,
                title=title,
                save_path=heatmap_path,
            )

            np.savez(
                result_path,
                x=x_np,
                t=t_np,
                u_true=usol_np,
                u_pred=u_pred,
                rel_l2=rel_l2,
                mse=mse,
                theta_file=str(path),
                architecture=np.asarray(arch, dtype=np.int64),
            )

            summary.append({
                "theta_file": str(path),
                "theta_size": int(theta.size),
                "architecture": "-".join(map(str, arch)),
                "rel_l2": rel_l2,
                "mse": mse,
                "heatmap": str(heatmap_path),
                "result_npz": str(result_path),
            })

            print(f"[OK] {path}")
            print(f"     theta_size={theta.size}, arch={arch}")
            print(f"     relL2={rel_l2:.3e}, mse={mse:.3e}")
            print(f"     heatmap -> {heatmap_path}")
            print(f"     result  -> {result_path}")

        except Exception as e:
            print(f"[SKIP] {path}")
            print(f"  {type(e).__name__}: {e}")

    summary_csv = outdir / "summary.csv"
    with open(summary_csv, "w") as f:
        f.write("theta_file,theta_size,architecture,rel_l2,mse,heatmap,result_npz\n")
        for row in summary:
            f.write(
                f"{row['theta_file']},"
                f"{row['theta_size']},"
                f"{row['architecture']},"
                f"{row['rel_l2']:.12e},"
                f"{row['mse']:.12e},"
                f"{row['heatmap']},"
                f"{row['result_npz']}\n"
            )

    print(f"\nSaved summary -> {summary_csv}")
    return summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "paths",
        nargs="*",
        default=["."],
        help="Theta files, folders, or glob patterns. Default: current folder."
    )
    parser.add_argument(
        "--mat",
        default="data/burgers.mat",
        help="Path to burgers.mat"
    )
    parser.add_argument(
        "--outdir",
        default="burgers_theta_heatmaps",
        help="Output folder"
    )

    # Jupyter/IPython injects extra args like --f=kernel.json.
    # parse_known_args() ignores those instead of crashing.
    args, unknown = parser.parse_known_args()

    if unknown:
        print(f"Ignoring unknown Jupyter/IPython args: {unknown}")

    process_all(args.paths, mat_path=args.mat, outdir=args.outdir)


if __name__ == "__main__":
    main()

Ignoring unknown Jupyter/IPython args: ['--f=/projects/549120ce-e7e0-45e0-a453-c9903649fea2/.local/share/jupyter/runtime/kernel-v36876bb97facd3e8be28ab25e1a61a5d62f3839cc.json']
Loading Burgers data: data/burgers.mat
Data: t=(201,), x=(512,), usol=(201, 512), nu=0.003183098861837907
Found 44 theta candidate files.
[SKIP] ./1/theta_sqp_transport_best.npy
  ValueError: Cannot infer architecture for ./1/theta_sqp_transport_best.npy. Theta size = 1981. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./1/theta_sqp_transport_best_deterministic.npy
  ValueError: Cannot infer architecture for ./1/theta_sqp_transport_best_deterministic.npy. Theta size = 1981. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./1/theta_sqp_transport_final.npy
  ValueError: Cannot infer architecture for ./1/theta_sqp_transport_final.npy. Theta size = 1981. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./1/theta_sqp_transport_final_deterministic.npy
  ValueError: Cannot infer architectur

2026-05-28 12:56:59.784774: W external/xla/xla/service/platform_util.cc:220] unable to create StreamExecutor for CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


[OK] ./burgers_adam_theta.npy
     theta_size=5301, arch=[2, 50, 50, 50, 1]
     relL2=3.983e-02, mse=5.971e-04
     heatmap -> burgers_theta_heatmaps/burgers_adam_theta_heatmap.png
     result  -> burgers_theta_heatmaps/burgers_adam_theta_result.npz
[SKIP] ./moose/theta_sqp_pinn_exp2_block_anchor_scaled.npy
  ValueError: Cannot infer architecture for ./moose/theta_sqp_pinn_exp2_block_anchor_scaled.npy. Theta size = 2876. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./theta_adam_pinn_ntc.npy
  ValueError: Cannot infer architecture for ./theta_adam_pinn_ntc.npy. Theta size = 2876. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./theta_adam_toy_multiphysics.npy
  ValueError: Cannot infer architecture for ./theta_adam_toy_multiphysics.npy. Theta size = 7926. Add it to ARCH_BY_THETA_SIZE or ARCH_BY_BASENAME.
[SKIP] ./theta_alm_burgers_no_bcder.npy
  ValueError: Cannot infer architecture for ./theta_alm_burgers_no_bcder.npy. Theta size = 1981. Add it to ARCH_BY_THETA_S

: 

Found 5 .npz files.
[SKIP] ./1/advection_c40_data.npz
"Could not identify keys in ./1/advection_c40_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./1/advection_c60_data.npz
"Could not identify keys in ./1/advection_c60_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./1/advection_wave_data.npz
"Could not identify keys in ./1/advection_wave_data.npz. Available keys: ['u_ref', 't_star', 'x_star', 'c', 'T', 'L']"
[SKIP] ./moose/reference_data/no_adv_320x256/single_T_no_adv_320x256_reference.npz
"Could not identify keys in ./moose/reference_data/no_adv_320x256/single_T_no_adv_320x256_reference.npz. Available keys: ['x', 'y', 'time', 'T', 'phi', 'fuel_nodes', 'coolant_nodes', 'interface_nodes', 'params']"
[SKIP] ./moose/sqp_dataset.npz
"Could not identify keys in ./moose/sqp_dataset.npz. Available keys: ['t', 'x', 'y', 'phi', 'T', 'phi_fuel', 'Ts', 'Tf', 'T_interface']"
Saved summary to heatmaps/summary.csv
